In [ ]:
import carla
import logging
import time
import math
import numpy as np
import cv2
import queue
import open3d as o3d


In [66]:
CARLA_HOST = "localhost"
CARLA_PORT = 2000
CARLA_TIMEOUT = 10.0

logging.basicConfig(
    level=logging.INFO, 
    format="%(asctime)s [%(levelname)s] %(name)s: %(message)s",
)

logger =  logging.getLogger(__name__)

CAMERA_POS_Z = 3
CAMERA_POS_X = 0

camera_init_trans = carla.Transform(carla.Location(z = CAMERA_POS_Z, x = CAMERA_POS_X))

IMAGE_WIDTH = 1000
IMAGE_HEIGHT = 700

RADAR_RANGE = 100.0
RADAR_HORIZONTAL_FOV = 90.0
RADAR_VERTICAL_FOV = 20.0

POINT_QUEUE = queue.Queue(maxsize=1)

In [41]:
def connect_to_carla() -> tuple[carla.Client, carla.World]:

    client = carla.Client(CARLA_HOST, CARLA_PORT)
    client.set_timeout(CARLA_TIMEOUT)
    world = client.get_world()

    if not world:

        raise RuntimeError(
            "Carla was not able to return a valid world\n"
        )
    
    logger.info("Connection to Carla established")
    return client, world

In [42]:
def define_spawn_point(world: carla.World) -> carla.Transform:

    spawn_points = world.get_map().get_spawn_points()

    spawn_point = spawn_points[0]

    logger.info("Spawn point defined.")
    return spawn_point

In [43]:
def spawn_ego_vehicle(world: carla.World, spawn_point: carla.Transform) -> carla.Vehicle:

    vehicle_bp = world.get_blueprint_library().filter("vehicle.*")

    vehicle =  world.spawn_actor(
        vehicle_bp[0],
        spawn_point
    )

    logger.info("Ego vehicle spawned.")

    return vehicle

In [44]:
def ego_vehicle_autopilot(vehicle: carla.Vehicle, autopilot_status: bool):

    vehicle.set_autopilot(
        autopilot_status,
    )

    logger.info(f"Autopilot set to {autopilot_status}")
        

In [58]:
def spawn_radar(vehicle: carla.Vehicle, world: carla.World, spawn_point: carla.Transform) -> carla.Sensor:

    radar_bp = world.get_blueprint_library().find("sensor.other.radar")

    radar_bp.set_attribute(
        "horizontal_fov",
        str(RADAR_HORIZONTAL_FOV)
    )

    radar_bp.set_attribute(
        "vertical_fov",
        str(RADAR_VERTICAL_FOV)
    )

    radar_bp.set_attribute(
        "range",
        str(RADAR_RANGE)
    )

    radar_bp.set_attribute(
        "points_per_second",
        "1500"
    )

    radar_bp.set_attribute(
        "sensor_tick",
        "0.05"
    )
    if not radar_bp:
        raise RuntimeError(
            "Carla was not able to provide a valid blueprint for radar."
        )
    
    radar = world.spawn_actor(radar_bp, camera_init_trans, attach_to = vehicle)
    logger.info(f"Radar was attached to vehicle with the following blueprint {radar_bp}")
    return radar

In [46]:
def radar_callback(radar_data):

    detections = []

    for detection in radar_data:

        detections.append({
            "velocity": detection.velocity,
            "depth": detection.depth,
            "azimuth": detection.azimuth,
            "altitude": detection.altitude
        })

    if POINT_QUEUE.full():

        try:
            POINT_QUEUE.get_nowait()
        except queue.Empty:
            pass

    POINT_QUEUE.put(detections)

In [47]:
def visualize_radar(detections):

    image = np.zeros(
        (IMAGE_HEIGHT, IMAGE_WIDTH, 3),
        dtype=np.uint8
    )

    center_x = IMAGE_WIDTH // 2
    center_y = IMAGE_HEIGHT - 50

    max_radius = IMAGE_HEIGHT - 100

    # Radar range rings
    for distance in [20, 40, 60, 80, 100]:

        radius = int(
            (distance / RADAR_RANGE) * max_radius
        )

        cv2.circle(
            image,
            (center_x, center_y),
            radius,
            (50, 50, 50),
            1
        )

        cv2.putText(
            image,
            f"{distance} m",
            (center_x + 8, center_y - radius),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.4,
            (150, 150, 150),
            1
        )

    # Vehicle position
    cv2.circle(
        image,
        (center_x, center_y),
        8,
        (255, 255, 255),
        -1
    )

    cv2.putText(
        image,
        "VEHICLE",
        (center_x + 12, center_y + 5),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.5,
        (255, 255, 255),
        1
    )

    # Radar detections
    for detection in detections:

        depth = detection["depth"]
        azimuth = detection["azimuth"]
        velocity = detection["velocity"]

        # Convert polar coordinates to image coordinates.
        radius = int(
            (depth / RADAR_RANGE) * max_radius
        )

        x = int(
            center_x + radius * math.sin(azimuth)
        )

        y = int(
            center_y - radius * math.cos(azimuth)
        )

        # Color based on radial velocity.
        if velocity < -0.5:
            color = (0, 0, 255)       # Approaching
        elif velocity > 0.5:
            color = (255, 0, 0)       # Receding
        else:
            color = (0, 255, 0)       # Nearly stationary

        cv2.circle(
            image,
            (x, y),
            4,
            color,
            -1
        )

    # Detection count
    cv2.putText(
        image,
        f"Detections: {len(detections)}",
        (20, 30),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.7,
        (255, 255, 255),
        2
    )

    cv2.imshow("CARLA - Live Radar", image)

In [62]:
def main():

    client, world = connect_to_carla()
    spawn_point = define_spawn_point(world)
    actors = world.get_actors().filter("vehicle.*")
    for vehicle in actors:
        vehicle.destroy()
    vehicle = spawn_ego_vehicle(world, spawn_point)
    ego_vehicle_autopilot(vehicle, True)
    radar = spawn_radar(vehicle, world, spawn_point)
    radar.listen(radar_callback)
    
    while True:
    
        try:
            detections = POINT_QUEUE.get(timeout=2.0)

        except queue.Empty:
            detections = []

        visualize_radar(detections)

        if detections:

            print(
                f"\nDetected objects: {len(detections)}"
            )

            for index, detection in enumerate(detections[:5]):

                print(
                    f"#{index + 1} | "
                    f"Distance: {detection['depth']:.2f} m | "
                    f"Azimuth: {math.degrees(detection['azimuth']):.2f} deg | "
                    f"Velocity: {detection['velocity']:.2f} m/s"
                )

        if cv2.waitKey(1) & 0xFF == ord("q"):
            break
        
    if radar is not None:
        radar.stop()
        radar.destroy()
    
    if vehicle is not None:
        vehicle.destroy()
    
    cv2.destroyAllWindows()

    print("Radar cleanup completed.")
    

In [67]:
if __name__ == "__main__":
    main()

2026-11-10 23:37:50,859 [INFO] __main__: Connection to Carla established
2026-11-10 23:37:50,987 [INFO] __main__: Spawn point defined.
2026-11-10 23:37:51,037 [INFO] __main__: Ego vehicle spawned.
2026-11-10 23:37:51,049 [INFO] __main__: Autopilot set to True
2026-11-10 23:37:51,080 [INFO] __main__: Radar was attached to vehicle with the following blueprint ActorBlueprint(id=sensor.other.radar,tags=[radar, sensor, other])



Detected objects: 21
#1 | Distance: 39.70 m | Azimuth: 25.04 deg | Velocity: 2901.74 m/s
#2 | Distance: 39.54 m | Azimuth: 19.70 deg | Velocity: 3132.55 m/s
#3 | Distance: 66.93 m | Azimuth: -22.43 deg | Velocity: 4168.94 m/s
#4 | Distance: 76.01 m | Azimuth: -0.61 deg | Velocity: 3915.58 m/s
#5 | Distance: 37.78 m | Azimuth: 12.02 deg | Velocity: 3494.62 m/s

Detected objects: 23
#1 | Distance: 89.43 m | Azimuth: 2.43 deg | Velocity: -0.11 m/s
#2 | Distance: 55.71 m | Azimuth: -28.46 deg | Velocity: -0.13 m/s
#3 | Distance: 73.07 m | Azimuth: -20.48 deg | Velocity: 0.10 m/s
#4 | Distance: 87.57 m | Azimuth: 42.23 deg | Velocity: -0.02 m/s
#5 | Distance: 46.36 m | Azimuth: 16.55 deg | Velocity: 0.23 m/s

Detected objects: 20
#1 | Distance: 92.99 m | Azimuth: -10.52 deg | Velocity: -0.08 m/s
#2 | Distance: 74.21 m | Azimuth: -3.13 deg | Velocity: -0.15 m/s
#3 | Distance: 91.67 m | Azimuth: -7.10 deg | Velocity: -0.19 m/s
#4 | Distance: 52.24 m | Azimuth: 0.20 deg | Velocity: -0.34 m/s
